# 損害サービス（協定業務）AI高度化ハンズオン

## このノートブックでやること

損害保険の**協定業務**（修理工場と修理内容・金額を合意するプロセス）を題材に、
Snowflake Cortex の AI 関数で次の3つを実現します。

| Step | 内容 | 使う機能 |
|---|---|---|
| Step 0 | 受領した事故車画像をセル内で確認 | `session.file.get` + `IPython.display.Image` |
| Step 1 | 事故車画像から損傷をAI判定 | `AI_COMPLETE`（Vision） |
| Step 2 | 見積書PDFから明細を構造化 | `AI_EXTRACT`（テーブル抽出） |
| Step 3 | 3層突合で不正を検知 | Dynamic Tables |

## 業務背景

技術アジャスターは慢性的に不足しており、1人が月100件以上の見積を目視で査定しています。
修理費の水増しや架空計上を見逃すと保険金の過払いが発生しますが、
全件を人手で精査するのは現実的ではありません。

そこで **画像・PDF・参照マスタの3点を突き合わせて機械的に異常を検出し、
アジャスターが見るべき案件だけを浮かび上がらせる** のがこのハンズオンのゴールです。

> データは全て架空のデモ用データです。実在の人物・企業・団体とは一切関係ありません。

In [ ]:
%%sql -r dataframe_1
-- コンテキストを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE INSURANCE_CLAIMS_WH;
USE DATABASE INSURANCE_CLAIMS_DB;
USE SCHEMA RAW;

-- ステージ上のファイルを確認（画像10件・PDF4件）
SELECT
    'CLAIM_IMAGES_STAGE' AS STAGE_NAME,
    RELATIVE_PATH,
    ROUND(SIZE/1024) AS SIZE_KB
FROM DIRECTORY(@CLAIM_IMAGES_STAGE)
UNION ALL
SELECT
    'DOCS_STAGE',
    RELATIVE_PATH,
    ROUND(SIZE/1024)
FROM DIRECTORY(@DOCS_STAGE)
ORDER BY STAGE_NAME, RELATIVE_PATH;

---
# Step 0: 受領した事故車画像を確認する

AI に分析させる前に、**まず人間が画像を見る**。これが協定業務の出発点です。

`session.file.get()` でステージ上の画像をローカルにダウンロードし、
`IPython.display.Image` でノートブック内に表示します。

なおステージが `ENCRYPTION = (TYPE='SNOWFLAKE_SSE')` で作られていることが前提です
（`setup.sql` で設定済み）。クライアントサイド暗号化のステージでは `file.get` できません。

In [ ]:
# 案件一覧に事故車画像を添えて表示する
import os, tempfile
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image, HTML

session = get_active_session()
df = session.sql("""
    SELECT
        c.CLAIM_ID,
        c.INSURED_NAME,
        v.MAKER || ' ' || v.MODEL_NAME AS VEHICLE,
        c.ACCIDENT_TYPE,
        i.IMAGE_PATH,
        c.ESTIMATED_TOTAL_YEN,
        c.REPAIR_SHOP_NAME
    FROM CLM_CLAIMS c
    JOIN CLM_IMAGES i
        ON c.CLAIM_ID = i.CLAIM_ID AND i.IMAGE_TYPE LIKE '事故%'
    JOIN REPAIR_REFERENCE.VEHICLES v
        ON c.VEHICLE_ID = v.VEHICLE_ID
    ORDER BY c.CLAIM_ID
""").to_pandas()

with tempfile.TemporaryDirectory() as tmp_dir:
    for _, row in df.iterrows():
        session.file.get(
            f"@INSURANCE_CLAIMS_DB.RAW.CLAIM_IMAGES_STAGE/{row['IMAGE_PATH']}",
            tmp_dir
        )
        local_path = os.path.join(tmp_dir, os.path.basename(row['IMAGE_PATH']))
        print(f"\n{'='*60}")
        print(f"CLAIM_ID: {row['CLAIM_ID']}  |  被保険者: {row['INSURED_NAME']}")
        print(f"車種: {row['VEHICLE']}  |  事故類型: {row['ACCIDENT_TYPE']}")
        print(f"見積額: {row['ESTIMATED_TOTAL_YEN']:,}円  |  修理工場: {row['REPAIR_SHOP_NAME']}")
        display(Image(filename=local_path, width=400))

In [ ]:
-- FILE 型のユーティリティ関数でファイル属性を確認する
-- Directory Table から TO_FILE(FILE_URL) を作るパターン
SELECT
    RELATIVE_PATH                           AS "パス",
    FL_GET_CONTENT_TYPE(TO_FILE(FILE_URL))  AS "MIMEタイプ",
    FL_GET_SIZE(TO_FILE(FILE_URL))          AS "バイト数",
    FL_GET_STAGE(TO_FILE(FILE_URL))         AS "ステージ"
FROM DIRECTORY(@CLAIM_IMAGES_STAGE)
WHERE RELATIVE_PATH LIKE 'claims/%'
ORDER BY RELATIVE_PATH;

## 新車状態と事故車を並べて比較する

アジャスターは「元がどうだったか」を頭に置いて損傷を判断します。
画像を2列並べれば、その比較作業をそのまま画面上で再現できます。

このあと AI が出す判定結果が妥当かどうか、**参加者自身の目で検証できる**ように
このセルを AI 分析の直前に置いています。

In [ ]:
# 新車参考画像（左）と事故車画像（右）を並べて表示
import os, tempfile
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image, HTML

session = get_active_session()
df = session.sql("""
    SELECT
        c.CLAIM_ID,
        v.MAKER || ' ' || v.MODEL_NAME AS VEHICLE,
        ref.IMAGE_PATH AS REF_PATH,
        acc.IMAGE_PATH AS ACC_PATH,
        c.ACCIDENT_DESCRIPTION
    FROM CLM_CLAIMS c
    JOIN CLM_IMAGES acc
        ON c.CLAIM_ID = acc.CLAIM_ID AND acc.IMAGE_TYPE LIKE '事故%'
    JOIN CLM_IMAGES ref
        ON c.CLAIM_ID = ref.CLAIM_ID AND ref.IMAGE_TYPE LIKE '参考%'
    JOIN REPAIR_REFERENCE.VEHICLES v
        ON c.VEHICLE_ID = v.VEHICLE_ID
    ORDER BY c.CLAIM_ID
""").to_pandas()

with tempfile.TemporaryDirectory() as tmp_dir:
    for _, row in df.iterrows():
        session.file.get(
            f"@INSURANCE_CLAIMS_DB.RAW.CLAIM_IMAGES_STAGE/{row['REF_PATH']}",
            tmp_dir
        )
        session.file.get(
            f"@INSURANCE_CLAIMS_DB.RAW.CLAIM_IMAGES_STAGE/{row['ACC_PATH']}",
            tmp_dir
        )
        ref_local = os.path.join(tmp_dir, os.path.basename(row['REF_PATH']))
        acc_local = os.path.join(tmp_dir, os.path.basename(row['ACC_PATH']))
        print(f"\n{'='*60}")
        print(f"CLAIM_ID: {row['CLAIM_ID']}  |  車種: {row['VEHICLE']}")
        print(f"事故状況: {row['ACCIDENT_DESCRIPTION']}")
        print(f"\n【新車参考】")
        display(Image(filename=ref_local, width=400))
        print(f"【事故車】")
        display(Image(filename=acc_local, width=400))

---
# Step 1: 事故車画像をAIで損傷判定する

`AI_COMPLETE` に Vision 対応モデルと **FILE 型**を渡すと、画像を解析できます。
`PROMPT('... {0}', TO_FILE(...))` の `{0}` が画像のプレースホルダです。

`TO_FILE()` はステージ上のファイルを SQL の値として扱うための型です。
結果グリッドにサムネイルとして描画されるわけではありませんが（Step 0 の画像表示は
Python で行っています）、**AI 関数の入力としてはこの型をそのまま渡せます**。

`response_format` に JSON スキーマを渡すことで、出力が OBJECT 型で返り、
後続の SQL でそのまま扱えます（自由文をパースする必要がありません）。

> **左右の判定について**
> 自動車部品の「左/右」は車両基準（運転席から前方を向いた状態）で決まります。
> 前方から撮った写真では車両の右側が画面の左に写るため、見た目とは逆になります。
> プロンプトでこの定義を明示し、Step 3 の突合でも左右の表記差は同一と見なしています。

> 東京リージョンでは Vision モデルが未提供の場合があります。
> `setup.sql` の `CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'` が有効である必要があります。
> `claude-sonnet-4-5` が使えない場合は `pixtral-large` / `llama4-maverick` を試してください。

In [ ]:
%%sql -r dataframe_5
-- 事故車画像から損傷部位・程度・推奨修理方法を構造化して抽出する
CREATE OR REPLACE TABLE STAGING.DAMAGE_ASSESSMENT AS
SELECT
    c.CLAIM_ID,
    i.IMAGE_PATH,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        PROMPT(
            'あなたは損害保険会社の技術アジャスターです。事故車の画像を分析してください。
【指示】
- 画像から視認できる損傷のみを挙げてください。推測で部位を追加しないこと。
- 部品名は日本語の一般的な自動車部品名称で答えてください。
- 修理方法は板金修正で足りるか取替が必要かを判断してください。
- 左右は「車両基準」で判定してください。運転席から前方を向いた状態での左右であり、
  写真を見たときの見た目の左右とは逆になる場合があります。判断できない場合は
  左右を付けずに部品名のみを答えてください。
【車両情報】
車種: {0}
事故状況: {1}
画像: {2}',
            v.MAKER || ' ' || v.MODEL_NAME || ' (' || v.MODEL_CODE || ')',
            c.ACCIDENT_DESCRIPTION,
            TO_FILE('@CLAIM_IMAGES_STAGE', i.IMAGE_PATH)
        ),
        {'response_format': {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'damaged_parts': {
                        'type': 'array',
                        'items': {'type': 'object', 'properties': {
                            'part_name':     {'type': 'string'},
                            'damage_type':   {'type': 'string'},
                            'severity':      {'type': 'string'},
                            'repair_method': {'type': 'string'}
                        }}
                    },
                    'overall_severity':        {'type': 'string'},
                    'estimated_total_min_yen': {'type': 'number'},
                    'estimated_total_max_yen': {'type': 'number'},
                    'summary':                 {'type': 'string'}
                }
            }
        }}
    ) AS ASSESSMENT
FROM CLM_CLAIMS c
JOIN CLM_IMAGES i
    ON c.CLAIM_ID = i.CLAIM_ID AND i.IMAGE_TYPE LIKE '事故%'
JOIN REPAIR_REFERENCE.VEHICLES v
    ON c.VEHICLE_ID = v.VEHICLE_ID;

SELECT * FROM STAGING.DAMAGE_ASSESSMENT ORDER BY CLAIM_ID;

### AI判定結果を明細行に展開する

`ASSESSMENT` は OBJECT 型なので、`LATERAL FLATTEN` で
損傷部位の配列を1部位1行に展開します。
これが後段で見積明細と突き合わせる基礎データになります。

In [ ]:
%%sql -r dataframe_6
-- 損傷部位を1行1部位に展開
CREATE OR REPLACE TABLE ANALYTICS.DT_DAMAGE_ASSESSMENT AS
SELECT
    a.CLAIM_ID,
    f.INDEX + 1                                  AS PART_SEQ,
    f.VALUE:part_name::STRING                     AS AI_PART_NAME,
    f.VALUE:damage_type::STRING                   AS AI_DAMAGE_TYPE,
    f.VALUE:severity::STRING                      AS AI_SEVERITY,
    f.VALUE:repair_method::STRING                 AS AI_REPAIR_METHOD,
    a.ASSESSMENT:overall_severity::STRING         AS OVERALL_SEVERITY,
    a.ASSESSMENT:estimated_total_min_yen::NUMBER  AS AI_MIN_YEN,
    a.ASSESSMENT:estimated_total_max_yen::NUMBER  AS AI_MAX_YEN,
    a.ASSESSMENT:summary::STRING                  AS AI_SUMMARY
FROM STAGING.DAMAGE_ASSESSMENT a,
     LATERAL FLATTEN(input => a.ASSESSMENT:damaged_parts) f;

SELECT
    CLAIM_ID, PART_SEQ, AI_PART_NAME, AI_DAMAGE_TYPE, AI_SEVERITY, AI_REPAIR_METHOD
FROM ANALYTICS.DT_DAMAGE_ASSESSMENT
ORDER BY CLAIM_ID, PART_SEQ;

### 画像とAI判定を突き合わせて検証する

事故車画像と AI の判定サマリを横に並べます。
**AIが本当に画像を見て答えているか**を参加者がその場で確認できます。
ここで納得感を得られると、以降の自動検知の結果も信頼できるようになります。

In [ ]:
# 画像・AI判定サマリ・AI概算額を横並びで検証
import os, tempfile
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image, HTML

session = get_active_session()
df = session.sql("""
    SELECT
        d.CLAIM_ID,
        i.IMAGE_PATH,
        d.OVERALL_SEVERITY,
        COUNT(*)                       AS PART_CNT,
        LISTAGG(d.AI_PART_NAME, ' / ') AS AI_PARTS,
        MAX(d.AI_MIN_YEN)              AS AI_MIN,
        MAX(d.AI_MAX_YEN)              AS AI_MAX,
        MAX(c.ESTIMATED_TOTAL_YEN)     AS SHOP_EST,
        MAX(d.AI_SUMMARY)              AS SUMMARY
    FROM ANALYTICS.DT_DAMAGE_ASSESSMENT d
    JOIN CLM_IMAGES i
        ON d.CLAIM_ID = i.CLAIM_ID AND i.IMAGE_TYPE LIKE '事故%'
    JOIN CLM_CLAIMS c
        ON d.CLAIM_ID = c.CLAIM_ID
    GROUP BY d.CLAIM_ID, i.IMAGE_PATH, d.OVERALL_SEVERITY
    ORDER BY d.CLAIM_ID
""").to_pandas()

with tempfile.TemporaryDirectory() as tmp_dir:
    for _, row in df.iterrows():
        session.file.get(
            f"@INSURANCE_CLAIMS_DB.RAW.CLAIM_IMAGES_STAGE/{row['IMAGE_PATH']}",
            tmp_dir
        )
        local_path = os.path.join(tmp_dir, os.path.basename(row['IMAGE_PATH']))
        print(f"\n{'='*60}")
        print(f"CLAIM_ID: {row['CLAIM_ID']}  |  AI判定: {row['OVERALL_SEVERITY']}  |  検出部位数: {row['PART_CNT']}")
        print(f"AI検出部位: {row['AI_PARTS']}")
        print(f"AI概算: {row['AI_MIN']:,}円 〜 {row['AI_MAX']:,}円  |  工場見積額: {row['SHOP_EST']:,}円")
        print(f"AI所見: {row['SUMMARY']}")
        display(Image(filename=local_path, width=400))

---
# Step 2: 見積書PDFから明細を構造化する

修理工場から届く見積書はPDFです。ここから明細表を機械可読な形に落とします。

`AI_EXTRACT` の **テーブル抽出**を使います。ポイントは2つあります。

1. `column_ordering` に **文書に現れる全列を、現れる順に**列挙する。
   途中で打ち切ると値が隣の列にずれます（`部品単価` に `工賃` が入る等）。
2. 列名は文書内の見出しと同じ表記にする。`No.` のような連番列も省略しない。

まず入力となる見積書PDFを FILE 型で確認します。

In [ ]:
%%sql -r dataframe_8
-- 見積書PDFを FILE 型で確認
SELECT
    e.CLAIM_ID,
    c.REPAIR_SHOP_NAME                        AS "修理工場",
    TO_FILE('@DOCS_STAGE', e.FILE_PATH)       AS "見積書PDF",
    FL_GET_CONTENT_TYPE(TO_FILE('@DOCS_STAGE', e.FILE_PATH)) AS "MIMEタイプ",
    ROUND(FL_GET_SIZE(TO_FILE('@DOCS_STAGE', e.FILE_PATH))/1024) AS SIZE_KB,
    e.RECEIVED_AT                             AS "受領日時"
FROM CLM_ESTIMATES_RAW e
JOIN CLM_CLAIMS c
    ON e.CLAIM_ID = c.CLAIM_ID
ORDER BY e.CLAIM_ID;

In [ ]:
%%sql -r dataframe_9
-- 見積明細をAI_EXTRACTで構造化し、1行1明細のテーブルに落とす
CREATE OR REPLACE TABLE STAGING.ESTIMATE_DETAILS AS
WITH extracted AS (
    SELECT
        e.CLAIM_ID,
        AI_EXTRACT(
            file => TO_FILE('@DOCS_STAGE', e.FILE_PATH),
            responseFormat => {
                'schema': {
                    'type': 'object',
                    'properties': {
                        'shop_name':   {'description': '見積書を発行した修理工場名', 'type': 'string'},
                        'grand_total': {'description': '見積合計金額', 'type': 'string'},
                        'items': {
                            'description': '見積明細表',
                            'type': 'object',
                            'column_ordering': ['No.','区分','部品番号','作業/部品名','数量','部品単価','指数','工賃','部品代','小計'],
                            'properties': {
                                'No.':         {'description': 'No.', 'type': 'array'},
                                '区分':        {'description': '区分', 'type': 'array'},
                                '部品番号':    {'description': '部品番号', 'type': 'array'},
                                '作業/部品名': {'description': '作業/部品名', 'type': 'array'},
                                '数量':        {'description': '数量', 'type': 'array'},
                                '部品単価':    {'description': '部品単価', 'type': 'array'},
                                '指数':        {'description': '指数', 'type': 'array'},
                                '工賃':        {'description': '工賃', 'type': 'array'},
                                '部品代':      {'description': '部品代', 'type': 'array'},
                                '小計':        {'description': '小計', 'type': 'array'}
                            }
                        }
                    }
                }
            }
        ):response AS RES
    FROM CLM_ESTIMATES_RAW e
)
SELECT
    x.CLAIM_ID,
    x.RES:shop_name::STRING   AS SHOP_NAME,
    x.RES:grand_total::STRING AS GRAND_TOTAL_TEXT,
    f.INDEX + 1               AS LINE_NO,
    x.RES:items:"区分"[f.INDEX]::STRING                  AS OPERATION_TYPE,
    -- 部品番号の正規化: 抽出時に数字の 0 が英字 O と誤読されることがある。
    -- マスタの部品番号に英字 O は含まれないため、O→0 の置換は安全。
    REPLACE(UPPER(NULLIF(x.RES:items:"部品番号"[f.INDEX]::STRING, '—')), 'O', '0') AS PART_NUMBER,
    x.RES:items:"作業/部品名"[f.INDEX]::STRING           AS WORK_NAME,
    TRY_TO_NUMBER(REGEXP_REPLACE(x.RES:items:"数量"[f.INDEX]::STRING,     '[^0-9]', ''))  AS QUANTITY,
    TRY_TO_NUMBER(REGEXP_REPLACE(x.RES:items:"部品単価"[f.INDEX]::STRING, '[^0-9]', ''))  AS UNIT_PRICE_YEN,
    TRY_TO_DOUBLE(REGEXP_REPLACE(x.RES:items:"指数"[f.INDEX]::STRING,     '[^0-9.]', '')) AS LABOR_INDEX,
    TRY_TO_NUMBER(REGEXP_REPLACE(x.RES:items:"工賃"[f.INDEX]::STRING,     '[^0-9]', ''))  AS LABOR_COST_YEN,
    TRY_TO_NUMBER(REGEXP_REPLACE(x.RES:items:"小計"[f.INDEX]::STRING,     '[^0-9]', ''))  AS SUBTOTAL_YEN
FROM extracted x,
     LATERAL FLATTEN(input => x.RES:items:"区分") f;

-- 抽出結果を確認（21明細が正しく取れていること）
SELECT
    CLAIM_ID, LINE_NO, OPERATION_TYPE, PART_NUMBER, WORK_NAME,
    UNIT_PRICE_YEN, LABOR_INDEX, LABOR_COST_YEN
FROM STAGING.ESTIMATE_DETAILS
ORDER BY CLAIM_ID, LINE_NO;

---
# Step 3: 3層突合で不正を検知する

ここまでで3つの情報が構造化データとして揃いました。

| データ | 出所 |
|---|---|
| 損傷部位（画像から） | `ANALYTICS.DT_DAMAGE_ASSESSMENT` |
| 見積明細（PDFから） | `STAGING.ESTIMATE_DETAILS` |
| 定価・標準指数（マスタ） | `STAGING.PARTS_MASTER` |

これを突き合わせて3種類の異常を検出します。

| チェック | 判定条件 | 狙い |
|---|---|---|
| 部品単価 | 見積単価 > 定価 × **1.15** | 単価水増し |
| 作業指数 | 見積指数 > 標準指数 × **1.30** | 工賃水増し |
| 部品番号 | マスタに存在しない | 架空部品・型式違い |
| 画像整合 | 画像にない部位が明細にある | 架空計上 |

In [ ]:
%%sql -r dataframe_10
-- 明細1行ごとに定価・標準指数と突合する
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_PRICE_ANOMALY
    TARGET_LAG = '1 hour'
    WAREHOUSE = INSURANCE_CLAIMS_WH
    REFRESH_MODE = FULL
    COMMENT = '見積明細と参照マスタの突合結果'
AS
SELECT
    d.CLAIM_ID,
    d.LINE_NO,
    d.OPERATION_TYPE,
    d.PART_NUMBER,
    d.WORK_NAME,
    d.UNIT_PRICE_YEN,
    m.LIST_PRICE_YEN,
    ROUND((d.UNIT_PRICE_YEN / NULLIF(m.LIST_PRICE_YEN, 0) - 1) * 100, 1) AS PRICE_DIFF_PCT,
    d.LABOR_INDEX,
    CASE d.OPERATION_TYPE
        WHEN '板金' THEN m.BODYWORK_INDEX
        ELSE m.REPLACEMENT_INDEX
    END AS REFERENCE_INDEX,
    ROUND((d.LABOR_INDEX / NULLIF(
        CASE d.OPERATION_TYPE WHEN '板金' THEN m.BODYWORK_INDEX ELSE m.REPLACEMENT_INDEX END, 0
    ) - 1) * 100, 1) AS INDEX_DIFF_PCT,
    CASE
        -- マスタ未登録は型式違いか架空部品の可能性
        WHEN d.PART_NUMBER IS NOT NULL AND m.PART_NUMBER IS NULL
            THEN 'UNKNOWN_PART'
        -- 部品単価が定価の15%超
        WHEN d.UNIT_PRICE_YEN > m.LIST_PRICE_YEN * 1.15
            THEN 'ALERT_PRICE_OVER'
        -- 板金指数が標準の30%超
        WHEN d.OPERATION_TYPE = '板金' AND d.LABOR_INDEX > m.BODYWORK_INDEX * 1.30
            THEN 'ALERT_INDEX_OVER'
        ELSE 'OK'
    END AS CHECK_RESULT
FROM STAGING.ESTIMATE_DETAILS d
LEFT JOIN STAGING.PARTS_MASTER m
    ON d.PART_NUMBER = m.PART_NUMBER;

-- 異常のみを抽出
SELECT
    CLAIM_ID, LINE_NO, CHECK_RESULT, WORK_NAME, PART_NUMBER,
    UNIT_PRICE_YEN, LIST_PRICE_YEN, PRICE_DIFF_PCT,
    LABOR_INDEX, REFERENCE_INDEX, INDEX_DIFF_PCT
FROM ANALYTICS.DT_PRICE_ANOMALY
WHERE CHECK_RESULT <> 'OK'
ORDER BY CLAIM_ID, LINE_NO;

### 画像に写っていない部位が計上されていないか

単価も指数も正常な範囲でも、**そもそも壊れていない部位を計上する**手口があります。
これは価格マスタとの突合では検出できません。

そこで Step 1 の画像判定結果と見積明細を突き合わせ、
**画像側に対応する損傷が見当たらない明細**を洗い出します。
部品名は表記が揺れるため、AI に意味的な一致を判断させます。

In [ ]:
%%sql -r dataframe_11
-- 見積明細ごとに、画像判定結果に対応する損傷があるかをAIで照合する
CREATE OR REPLACE TABLE ANALYTICS.DT_IMAGE_CONSISTENCY AS
WITH ai_parts AS (
    SELECT CLAIM_ID, LISTAGG(AI_PART_NAME, '、') AS AI_PARTS
    FROM ANALYTICS.DT_DAMAGE_ASSESSMENT
    GROUP BY CLAIM_ID
),
-- 部品明細のみを対象にする（塗装・板金は部品行に紐づくため重複を避ける）
target AS (
    SELECT DISTINCT d.CLAIM_ID, d.PART_NUMBER, d.WORK_NAME
    FROM STAGING.ESTIMATE_DETAILS d
    WHERE d.OPERATION_TYPE = '部品' AND d.PART_NUMBER IS NOT NULL
)
SELECT
    t.CLAIM_ID,
    t.PART_NUMBER,
    t.WORK_NAME,
    a.AI_PARTS,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        '事故車画像のAI分析で検出された損傷部位は次の通りです。
【画像から検出された損傷部位】' || a.AI_PARTS ||
        '
見積書に計上されている部品「' || t.WORK_NAME || '」は、上記の損傷部位のいずれかに該当しますか。
表記の揺れ（例: ボンネットとフードは同一）は同一と見なしてください。
左右（左/右）の表記差は同一と見なしてください。単一の写真から左右を確定するのは
困難なため、前後・部位が一致していれば該当と判断してください。
該当するなら CONSISTENT、該当しないなら NOT_IN_IMAGE の1語のみを出力してください。',
        {'response_format': {'type': 'json', 'schema': {'type': 'object',
            'properties': {'judgement': {'type': 'string'}}}}}
    ):judgement::STRING AS JUDGEMENT
FROM target t
JOIN ai_parts a ON t.CLAIM_ID = a.CLAIM_ID;

-- 画像に見当たらない計上部品（架空計上の疑い）
SELECT
    CLAIM_ID, PART_NUMBER, WORK_NAME, AI_PARTS, JUDGEMENT
FROM ANALYTICS.DT_IMAGE_CONSISTENCY
WHERE JUDGEMENT = 'NOT_IN_IMAGE'
ORDER BY CLAIM_ID;

In [ ]:
%%sql -r dataframe_12
-- 案件単位のリスクスコア（0〜100）を算出する
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_FRAUD_RISK_SCORE
    TARGET_LAG = '1 hour'
    WAREHOUSE = INSURANCE_CLAIMS_WH
    REFRESH_MODE = FULL
    COMMENT = '案件別の不正リスクスコアとアラート内訳'
AS
WITH price_alerts AS (
    SELECT
        CLAIM_ID,
        COUNT_IF(CHECK_RESULT = 'ALERT_PRICE_OVER') AS CNT_PRICE_OVER,
        COUNT_IF(CHECK_RESULT = 'ALERT_INDEX_OVER') AS CNT_INDEX_OVER,
        COUNT_IF(CHECK_RESULT = 'UNKNOWN_PART')     AS CNT_UNKNOWN_PART,
        MAX(GREATEST(COALESCE(PRICE_DIFF_PCT, 0), COALESCE(INDEX_DIFF_PCT, 0))) AS MAX_DIFF_PCT
    FROM ANALYTICS.DT_PRICE_ANOMALY
    GROUP BY CLAIM_ID
),
image_alerts AS (
    SELECT CLAIM_ID, COUNT_IF(JUDGEMENT = 'NOT_IN_IMAGE') AS CNT_NOT_IN_IMAGE
    FROM ANALYTICS.DT_IMAGE_CONSISTENCY
    GROUP BY CLAIM_ID
)
SELECT
    c.CLAIM_ID,
    c.INSURED_NAME,
    c.REPAIR_SHOP_NAME,
    c.ESTIMATED_TOTAL_YEN,
    COALESCE(p.CNT_PRICE_OVER, 0)   AS CNT_PRICE_OVER,
    COALESCE(p.CNT_INDEX_OVER, 0)   AS CNT_INDEX_OVER,
    COALESCE(p.CNT_UNKNOWN_PART, 0) AS CNT_UNKNOWN_PART,
    COALESCE(i.CNT_NOT_IN_IMAGE, 0) AS CNT_NOT_IN_IMAGE,
    ROUND(COALESCE(p.MAX_DIFF_PCT, 0), 1) AS MAX_DIFF_PCT,
    -- 重み付け: 架空計上30点 / 単価水増し25点 / 指数水増し25点 / マスタ未登録15点
    LEAST(100,
          COALESCE(i.CNT_NOT_IN_IMAGE, 0) * 30
        + COALESCE(p.CNT_PRICE_OVER, 0)   * 25
        + COALESCE(p.CNT_INDEX_OVER, 0)   * 25
        + COALESCE(p.CNT_UNKNOWN_PART, 0) * 15
    ) AS RISK_SCORE,
    CASE
        WHEN LEAST(100,
                 COALESCE(i.CNT_NOT_IN_IMAGE, 0) * 30
               + COALESCE(p.CNT_PRICE_OVER, 0)   * 25
               + COALESCE(p.CNT_INDEX_OVER, 0)   * 25
               + COALESCE(p.CNT_UNKNOWN_PART, 0) * 15) >= 50 THEN '要精査'
        WHEN LEAST(100,
                 COALESCE(i.CNT_NOT_IN_IMAGE, 0) * 30
               + COALESCE(p.CNT_PRICE_OVER, 0)   * 25
               + COALESCE(p.CNT_INDEX_OVER, 0)   * 25
               + COALESCE(p.CNT_UNKNOWN_PART, 0) * 15) >= 20 THEN '要確認'
        ELSE '問題なし'
    END AS RISK_LEVEL
FROM RAW.CLM_CLAIMS c
LEFT JOIN price_alerts p ON c.CLAIM_ID = p.CLAIM_ID
LEFT JOIN image_alerts i ON c.CLAIM_ID = i.CLAIM_ID;

SELECT * FROM ANALYTICS.DT_FRAUD_RISK_SCORE ORDER BY RISK_SCORE DESC;

In [ ]:
%%sql -r dataframe_13
-- 工場別の実績を集計する（常習性の把握）
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_SHOP_PERFORMANCE
    TARGET_LAG = '1 hour'
    WAREHOUSE = INSURANCE_CLAIMS_WH
    REFRESH_MODE = FULL
    COMMENT = '修理工場別のアラート実績'
AS
SELECT
    s.SHOP_ID,
    s.SHOP_NAME,
    s.PREFECTURE,
    s.CERTIFIED_LEVEL,
    s.HISTORICAL_ALERT_RATE,
    COUNT(r.CLAIM_ID)                                AS CLAIM_CNT,
    SUM(r.ESTIMATED_TOTAL_YEN)                       AS TOTAL_ESTIMATED_YEN,
    ROUND(AVG(r.RISK_SCORE), 1)                      AS AVG_RISK_SCORE,
    MAX(r.RISK_SCORE)                                AS MAX_RISK_SCORE,
    COUNT_IF(r.RISK_SCORE >= 50)                     AS CNT_NEEDS_REVIEW,
    ROUND(COUNT_IF(r.RISK_SCORE >= 50)
          / NULLIF(COUNT(r.CLAIM_ID), 0) * 100, 1)  AS ALERT_RATE_PCT
FROM RAW.CLM_REPAIR_SHOPS s
LEFT JOIN ANALYTICS.DT_FRAUD_RISK_SCORE r
    ON s.SHOP_NAME = r.REPAIR_SHOP_NAME
GROUP BY ALL
ORDER BY AVG_RISK_SCORE DESC NULLS LAST;

SELECT * FROM ANALYTICS.DT_SHOP_PERFORMANCE;

### 最終アウトプット: アジャスターが見るべき案件

不正の疑いがある案件を、**画像付きで**一覧化します。
アジャスターはこの画面から直接、根拠となる画像を確認して判断できます。

In [ ]:
# 要精査案件を画像付きで一覧表示（アジャスターの作業画面イメージ）
import os, tempfile
from snowflake.snowpark.context import get_active_session
from IPython.display import display, Image, HTML

session = get_active_session()
df = session.sql("""
    SELECT
        r.CLAIM_ID,
        r.RISK_LEVEL,
        r.RISK_SCORE,
        i.IMAGE_PATH,
        r.INSURED_NAME,
        r.REPAIR_SHOP_NAME,
        r.ESTIMATED_TOTAL_YEN,
        r.CNT_NOT_IN_IMAGE,
        r.CNT_PRICE_OVER,
        r.CNT_INDEX_OVER,
        r.CNT_UNKNOWN_PART,
        r.MAX_DIFF_PCT
    FROM ANALYTICS.DT_FRAUD_RISK_SCORE r
    JOIN RAW.CLM_IMAGES i
        ON r.CLAIM_ID = i.CLAIM_ID AND i.IMAGE_TYPE LIKE '事故%'
    ORDER BY r.RISK_SCORE DESC
""").to_pandas()

with tempfile.TemporaryDirectory() as tmp_dir:
    for _, row in df.iterrows():
        session.file.get(
            f"@INSURANCE_CLAIMS_DB.RAW.CLAIM_IMAGES_STAGE/{row['IMAGE_PATH']}",
            tmp_dir
        )
        local_path = os.path.join(tmp_dir, os.path.basename(row['IMAGE_PATH']))
        print(f"\n{'='*60}")
        print(f"【{row['RISK_LEVEL']}】 CLAIM_ID: {row['CLAIM_ID']}  |  リスクスコア: {row['RISK_SCORE']}")
        print(f"被保険者: {row['INSURED_NAME']}  |  修理工場: {row['REPAIR_SHOP_NAME']}")
        print(f"見積額: {row['ESTIMATED_TOTAL_YEN']:,}円")
        print(f"架空計上疑い: {row['CNT_NOT_IN_IMAGE']}  |  単価超過: {row['CNT_PRICE_OVER']}  |  指数超過: {row['CNT_INDEX_OVER']}  |  マスタ未登録: {row['CNT_UNKNOWN_PART']}  |  最大乖離率: {row['MAX_DIFF_PCT']}%")
        display(Image(filename=local_path, width=400))

---
# まとめ

## 検出された3つの不正パターン

| 案件 | 車種 | 手口 | 検出した根拠 |
|---|---|---|---|
| CLM-2024-0001 | プリウス | — | 提出画像に損傷が写っておらず、申告内容と不一致（再撮影依頼） |
| CLM-2024-0003 | ノート E13 | **架空計上** | 画像の損傷は右側フロント周辺なのにリヤバンパーフェイシア（車の後方部分のパーツ）を計上／マスタ未登録の部品番号 |
| CLM-2024-0004 | N-BOX JF5 | **単価水増し** | リヤバンパフェイス 46,800円 vs 定価 37,400円（+25%） |
| CLM-2024-0005 | ヤリスクロス | **指数水増し** | 左フロントフェンダ板金 指数9.5 vs 標準3.8（+150%） |

重要なのは、**3つの手口はそれぞれ別のチェックでしか捕まらない**という点です。
価格マスタとの突合だけでは架空計上は見抜けず、画像分析だけでは単価の妥当性は判断できません。
画像・PDF・マスタを同じプラットフォーム上で突き合わせられることが、この検知を成立させています。

## 使った機能

- **`session.file.get`** — ステージ上の画像をローカルにダウンロードし、`IPython.display.Image` でセル内に表示
- **`AI_COMPLETE`（Vision）** — 画像から損傷を構造化JSONで抽出
- **`response_format`** — 出力をOBJECT型に固定し、自由文のパースを不要にする
- **`AI_EXTRACT`（テーブル抽出）** — PDFの明細表を1行1明細に構造化
- **Dynamic Tables** — 突合ロジックを宣言的に定義し、データ更新に追随させる

## 実務での注意点

抽出結果はそのまま使えるとは限りません。今回も部品番号の
数字 `0` が英字 `O` と読まれるケースがあり、正規化を入れて初めて
マスタと結合できました。**抽出→正規化→結合** を一連の工程として
設計するのが実用上のポイントです。

## 次のステップ

| ファイル | 内容 |
|---|---|
| `src/02_cortex_search.sql` | 約款PDFを検索可能にする（Cortex Search） |
| `src/03_cortex_agent.sql` | 案件照会・不正分析・約款検索を横断するAIエージェント |